# Exploração dos dados e tratamento


In [1]:
from pathlib import Path
import pandas as pd
import diplo_grafo.config as cfg

# Ajuste para a pasta do projeto
base = cfg.SAMPLE_DATA_DIR

# Carrega os CSVs
cases = pd.read_csv(base / "cases.csv")
metadata = pd.read_csv(base / "metadata.csv")
data_dict = pd.read_csv(base / "data_dictionary.csv")

# Mostra as primeiras linhas
print("cases:")
print(cases.head())

#pega o texto de cada caso
cases_text = cases['case_text'].tolist()
print("Texts from cases:")
print(cases_text)


2026-09-08 18:35:05.027 | INFO     | diplo_grafo.config:<module>:11 - PROJ_ROOT path is: /home/lucas/Documents/NLP/Diplotokens-P1/project 1


cases:
   article_id   age        case_id  \
0  PMC5137649  44.0  PMC5137649_01   
1  PMC9387390  57.0  PMC9387390_01   
2  PMC3437073  55.0  PMC3437073_01   
3  PMC7102447  65.0  PMC7102447_01   
4  PMC6354154  30.0  PMC6354154_01   

                                           case_text  gender  
0  A 44-year-old woman presented with a 3-day his...  Female  
1  A 57-year-old man with no significant past med...    Male  
2  A 55-year-old male presented with a gradually ...    Male  
3  A 65-year-old male with a past medical history...    Male  
4  A 30-year-old nulligravida presented herself i...  Female  
Texts from cases:
['A 44-year-old woman presented with a 3-day history of right flank and lower quadrant abdominal pain associated with nausea and constipation. Her past medical, family and medication history were otherwise non-contributory and her physical examination was unremarkable. She underwent contrast enhanced computed tomography, demonstrating a 6cm cystic lesion between the

In [2]:
import re

PATTERNS = {
    # Idade e genero (ex: "A 44-year-old woman", "65-year-old male", "20-month-old boy")
    "patient_demographics": re.compile(
        r"(?:A|An)\s+(\d{1,3})-(year|month|day)-old\s+(male|female|man|woman|boy|girl)",
        re.IGNORECASE,
    ),
    
    # Pressao arterial (ex: "195/110 mmHg", "120/80")
    "blood_pressure": re.compile(
        r"(\d{2,3})/(\d{2,3})\s*(?:mmHg)?",
        re.IGNORECASE,
    ),
    
    # Valores de laboratorio com unidades (ex: "12,476.5ng/ml", "1.9 mg/dL", "66 mg/dL")
    "lab_measurement": re.compile(
        r"(\d+(?:,\d+)*(?:\.\d+)?)\s*(mg/dL|g/dL|ng/mL|U/L|iu/ml|mm/h|mmol/L|g%|fL)",
        re.IGNORECASE,
    ),
    
    # Tamanho de lesoes e medidas (ex: "6cm x 9cm", "9.5cm x 4.5cm x 2.0cm", "4 cm")
    "dimensions": re.compile(
        r"(\d+(?:\.\d+)?)\s*(cm|mm)(?:\s*x\s*(\d+(?:\.\d+)?)\s*(cm|mm))?",
        re.IGNORECASE,
    ),
    
    # Duracao e historico temporal (ex: "3-day history", "3-year history")
    "duration": re.compile(
        r"(\d+)-(day|week|month|year)\s+history",
        re.IGNORECASE,
    ),
    
    # Negacoes (ex: "denied fever", "no evidence of malignancy", "without rebound")
    "negation": re.compile(
        r"\b(denied|denies|no evidence of|negative for|free of|unremarkable|without)\b",
        re.IGNORECASE,
    ),
}

def extract_tokens(text):
    tokens = []

    for pattern_name, pattern in PATTERNS.items():
        for match in pattern.finditer(text):
            tokens.append({
                "tipo": pattern_name,
                "texto": match.group(0),
                "grupos": match.groups(),
                "inicio": match.start(),
                "fim": match.end(),
            })

    return sorted(tokens, key=lambda token: token["inicio"])


for index, text in enumerate(cases_text, start=1):
    tokens = extract_tokens(text)

    print(f"\n{'=' * 80}")
    print(f"CASO {index}")
    print("\nTexto:")
    print(text)

    print("\nTokens encontrados:")
    if tokens:
        for token in tokens:
            print(
                f"- {token['tipo']}: {token['texto']!r} "
                f"(posição {token['inicio']}–{token['fim']})"
            )
    else:
        print("Nenhum token encontrado.")




CASO 1

Texto:
A 44-year-old woman presented with a 3-day history of right flank and lower quadrant abdominal pain associated with nausea and constipation. Her past medical, family and medication history were otherwise non-contributory and her physical examination was unremarkable. She underwent contrast enhanced computed tomography, demonstrating a 6cm cystic lesion between the stomach and body/tail of the pancreas (Fig 1). She subsequently underwent EUS-FNA, which revealed normal pancreatic echotexture and a cyst measuring 6cm x 9cm that was free of internal septations or associated masses (Fig 2) but compressed the stomach. FNA of the cyst demonstrated no evidence of malignancy but did show the presence of extracellular mucin as well as a carcinoembryonic antigen (CEA) level of 12,476.5ng/ml and a carbohydrate antigen (CA) 19-9 level of 6iu/ml, suggesting the diagnosis of a mucinous pancreatic cystic neoplasm. The patient was therefore referred for surgical resection.   
A laparosc

In [3]:
# Sentence separator
SENTENCE_SEP_REGEX = r"(\. )"
re_sentence_sep = re.compile(SENTENCE_SEP_REGEX)

def find_separators(text):
    seps = []

    for match in re_sentence_sep.finditer(text):
        seps.append({
            "texto": match.group(0),
            "pos": match.start(),
        })

    return sorted(seps, key=lambda sep: sep["pos"])




for index, text in enumerate(cases_text, start=1):
    seps = find_separators(text)

    print(f"\n{'=' * 80}")
    print(f"CASO {index}")
    print("\nTexto:")
    print(text)

    print("\nSeparators encontrados:")
    if seps:
        for s in seps:
            p = s['pos']
            print(
                f"Trecho: '{text[p-5:p+5]}' "
                f"(pos: {p})"
            )
    else:
        print("Nenhum sep encontrado.")


CASO 1

Texto:
A 44-year-old woman presented with a 3-day history of right flank and lower quadrant abdominal pain associated with nausea and constipation. Her past medical, family and medication history were otherwise non-contributory and her physical examination was unremarkable. She underwent contrast enhanced computed tomography, demonstrating a 6cm cystic lesion between the stomach and body/tail of the pancreas (Fig 1). She subsequently underwent EUS-FNA, which revealed normal pancreatic echotexture and a cyst measuring 6cm x 9cm that was free of internal septations or associated masses (Fig 2) but compressed the stomach. FNA of the cyst demonstrated no evidence of malignancy but did show the presence of extracellular mucin as well as a carcinoembryonic antigen (CEA) level of 12,476.5ng/ml and a carbohydrate antigen (CA) 19-9 level of 6iu/ml, suggesting the diagnosis of a mucinous pancreatic cystic neoplasm. The patient was therefore referred for surgical resection.   
A laparosc

In [8]:
import re

token_sep_regex = re.compile(r"""
    \d+(?:,\d{3})*(?:\.\d+)?(?:ng/ml|iu/ml|mg/dl|cm|mm)?  # números com unidade grudada
  | [A-Za-z]+(?:-[A-Za-z]+)*                                # palavras, incl. hifenizadas (EUS-FNA)
  | [.,;:()]                                                 # pontuação, como token separado
""", re.VERBOSE)


def find_separators_tokens(text):
    seps = []

    for match in token_sep_regex.finditer(text):
        seps.append({
            "texto": match.group(0),
            "pos": match.start(),
        })

    return sorted(seps, key=lambda sep: sep["pos"])



for index, text in enumerate(cases_text, start=1):
    seps = find_separators(text)

    print(f"\n{'=' * 80}")
    print(f"CASO {index}")
    print("\nTexto:")
    print(text)

    tokens = find_separators_tokens(text)

    print("\nTokens encontrados:")
    if tokens:
        for t in tokens:
            print(f"- {t['texto']!r}")
    else:
        print("Nenhum token encontrado.")



CASO 1

Texto:
A 44-year-old woman presented with a 3-day history of right flank and lower quadrant abdominal pain associated with nausea and constipation. Her past medical, family and medication history were otherwise non-contributory and her physical examination was unremarkable. She underwent contrast enhanced computed tomography, demonstrating a 6cm cystic lesion between the stomach and body/tail of the pancreas (Fig 1). She subsequently underwent EUS-FNA, which revealed normal pancreatic echotexture and a cyst measuring 6cm x 9cm that was free of internal septations or associated masses (Fig 2) but compressed the stomach. FNA of the cyst demonstrated no evidence of malignancy but did show the presence of extracellular mucin as well as a carcinoembryonic antigen (CEA) level of 12,476.5ng/ml and a carbohydrate antigen (CA) 19-9 level of 6iu/ml, suggesting the diagnosis of a mucinous pancreatic cystic neoplasm. The patient was therefore referred for surgical resection.   
A laparosc